# Assignment 2 — Stage 1 Sentiment Classification

This notebook trains a neural classifier for movie reviews.

- `0` = negative
- `1` = positive

Only `train.csv` is used for model fitting. `public_test.csv` is used only for evaluation.

## Model structure

The supplied RNN tutorial demonstrates the common neural-language pipeline of token IDs, embeddings, neural processing, and output scores. For this small dataset, I use a simpler and faster neural text classifier:

1. tokenize each review;
2. map tokens to IDs;
3. average learned token embeddings with `EmbeddingBag`;
4. apply a hidden ReLU layer and dropout;
5. output two class scores.

This keeps the model reproducible on a CPU laptop while still learning neural word representations.

## Handling the small and imbalanced training set

- A stratified 80/20 split is used for training and validation.
- Class-weighted cross-entropy gives the minority negative class more influence.
- Dropout and AdamW weight decay reduce overfitting.
- Early stopping keeps the checkpoint with the lowest validation loss.
- An `<UNK>` token handles evaluation words not found in training.
- The public test set is never used for training.

## Key training techniques

- Optimizer: AdamW
- Learning rate: `0.003`
- Batch size: `16`
- Maximum epochs: `30`
- Early-stopping patience: `5`
- Embedding size: `128`
- Hidden layer size: `64`
- Dropout: `0.4`

In [ ]:
from __future__ import annotations
import json, random, re
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

train_df = pd.read_csv("train.csv")
public_test_df = pd.read_csv("public_test.csv")

print(train_df.shape, public_test_df.shape)
print(train_df["label"].value_counts().sort_index())

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")

def tokenize(text):
    return TOKEN_RE.findall(str(text).lower())

train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=0.2,
    random_state=SEED,
    stratify=train_df["label"],
)

counter = Counter()
for text in train_df.iloc[train_idx]["text"]:
    counter.update(tokenize(text))

vocab = {"<UNK>": 0}
for word, count in counter.most_common(12000):
    if count >= 2:
        vocab[word] = len(vocab)

def encode(text):
    ids = [vocab.get(word, 0) for word in tokenize(text)]
    return ids or [0]

print("Vocabulary size:", len(vocab))

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame["text"].tolist()
        self.labels = frame["label"].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return encode(self.texts[index]), self.labels[index]

def collate(batch):
    all_ids, offsets, labels = [], [0], []
    for ids, label in batch:
        all_ids.extend(ids)
        offsets.append(offsets[-1] + len(ids))
        labels.append(label)
    return (
        torch.tensor(all_ids, dtype=torch.long),
        torch.tensor(offsets[:-1], dtype=torch.long),
        torch.tensor(labels, dtype=torch.long),
    )

class SentimentEmbeddingBag(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=64, dropout=0.4):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embedding_dim, mode="mean")
        self.network = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, text, offsets):
        return self.network(self.embedding(text, offsets))

In [ ]:
train_loader = DataLoader(
    ReviewDataset(train_df.iloc[train_idx].reset_index(drop=True)),
    batch_size=16, shuffle=True, collate_fn=collate
)
validation_loader = DataLoader(
    ReviewDataset(train_df.iloc[val_idx].reset_index(drop=True)),
    batch_size=16, shuffle=False, collate_fn=collate
)
public_test_loader = DataLoader(
    ReviewDataset(public_test_df.reset_index(drop=True)),
    batch_size=32, shuffle=False, collate_fn=collate
)

model = SentimentEmbeddingBag(len(vocab))
counts = train_df.iloc[train_idx]["label"].value_counts().sort_index().to_numpy()
class_weights = len(train_idx) / (2.0 * counts)

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float32)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.001)

def evaluate(loader):
    model.eval()
    predictions, labels_all = [], []
    total_loss = 0.0
    with torch.no_grad():
        for text, offsets, labels in loader:
            logits = model(text, offsets)
            total_loss += criterion(logits, labels).item() * len(labels)
            predictions.extend(logits.argmax(1).tolist())
            labels_all.extend(labels.tolist())
    return (
        total_loss / len(labels_all),
        accuracy_score(labels_all, predictions),
        predictions,
        labels_all,
    )

In [ ]:
history = []
best_state = None
best_validation_loss = float("inf")
wait = 0

for epoch in range(1, 31):
    model.train()
    total_loss = 0.0

    for text, offsets, labels in train_loader:
        optimizer.zero_grad()
        logits = model(text, offsets)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    validation_loss, validation_accuracy, _, _ = evaluate(validation_loader)
    history.append({
        "epoch": epoch,
        "train_loss": total_loss / len(train_idx),
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
    })

    print(
        f"Epoch {epoch:02d}: "
        f"train_loss={history[-1]['train_loss']:.4f}, "
        f"val_loss={validation_loss:.4f}, "
        f"val_accuracy={validation_accuracy:.4f}"
    )

    if validation_loss < best_validation_loss - 0.0001:
        best_validation_loss = validation_loss
        best_state = {
            key: value.detach().clone()
            for key, value in model.state_dict().items()
        }
        wait = 0
    else:
        wait += 1
        if wait >= 5:
            print("Early stopping.")
            break

model.load_state_dict(best_state)

In [ ]:
history_df = pd.DataFrame(history)
display(history_df)

plt.figure(figsize=(7, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Training loss")
plt.plot(history_df["epoch"], history_df["validation_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.show()

In [ ]:
public_loss, public_accuracy, public_predictions, public_labels = evaluate(
    public_test_loader
)
public_confusion = confusion_matrix(public_labels, public_predictions)

print(f"Public-test accuracy: {public_accuracy:.4f}")
print(public_confusion)

ConfusionMatrixDisplay(
    confusion_matrix=public_confusion,
    display_labels=["Negative (0)", "Positive (1)"],
).plot()
plt.title("Public-test confusion matrix")
plt.show()

In [ ]:
predictions_df = pd.DataFrame({
    "id": public_test_df["id"],
    "predicted_label": public_predictions,
})
predictions_df.to_csv("public_test_predictions.csv", index=False)
display(predictions_df.head())

In [ ]:
checkpoint_dir = Path("model_checkpoint")
checkpoint_dir.mkdir(exist_ok=True)

config = {
    "vocab_size": len(vocab),
    "embedding_dim": 128,
    "hidden_dim": 64,
    "dropout": 0.4,
    "token_pattern": TOKEN_RE.pattern,
}

torch.save(best_state, checkpoint_dir / "model_state.pt")
with open(checkpoint_dir / "vocab.json", "w", encoding="utf-8") as file:
    json.dump(vocab, file, indent=2)
with open(checkpoint_dir / "config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)
with open(checkpoint_dir / "metrics.json", "w", encoding="utf-8") as file:
    json.dump({
        "public_test_accuracy": public_accuracy,
        "public_test_confusion_matrix": public_confusion.tolist(),
        "epochs_completed": len(history),
    }, file, indent=2)

print("Checkpoint saved.")

In [ ]:
# Reload test: this is the pattern Stage 2 should use without retraining.
with open("model_checkpoint/config.json", encoding="utf-8") as file:
    loaded_config = json.load(file)

reloaded_model = SentimentEmbeddingBag(
    loaded_config["vocab_size"],
    loaded_config["embedding_dim"],
    loaded_config["hidden_dim"],
    loaded_config["dropout"],
)
reloaded_model.load_state_dict(
    torch.load("model_checkpoint/model_state.pt", map_location="cpu")
)
reloaded_model.eval()
print("Checkpoint reloaded successfully.")

## Public-test evaluation

The final checkpoint achieved an accuracy of **0.6675**.

Confusion matrix, with true labels as rows and predicted labels as columns:

```text
[[119  81]
 [ 52 148]]
```

The public test set was used only for evaluation and prediction, not for training.

## Use of AI

Generative AI was used to help organize the notebook, explain the model, and review Python code. The model was trained only on the provided `train.csv`, while `public_test.csv` was used only for evaluation and predictions. I reviewed the code, outputs, and explanations before submission.